### Load Dataset

In [1]:
import os
from datasets import load_dataset

# Load the specific parquet file
test_dataset = load_dataset("parquet", data_files={'test': os.path.join('..', 'data', 'test-00000-of-00001.parquet')})
train_dataset = load_dataset("parquet", data_files={'train': os.path.join('..', 'data', 'train-00000-of-00001.parquet')})
validation_dataset = load_dataset("parquet", data_files={'validation': os.path.join('..', 'data', 'validation-00000-of-00001.parquet')})

test_df = test_dataset['test'].to_pandas()
train_df = train_dataset['train'].to_pandas()
validation_df = validation_dataset['validation'].to_pandas()

display(train_df)
display(test_df)
display(validation_df)

,text,intent
0,what are the steps for setting up direct depos...,108
1,how is a direct deposit set up,108
2,how would i go about setting up a direct deposit,108
3,tell me how to set up a direct deposit,108
4,how do i arrange a direct deposit into my savi...,108
...,...,...
10620,what percentage of species display cold bloode...,42
10621,what does it mean to be an alpha male,42
10622,what animals have alpha males,42
10623,why do males want to be alpha,42


,text,intent
0,how would you say fly in italian,61
1,what's the spanish word for pasta,61
2,how would they say butter in zambia,61
3,how do you say fast in spanish,61
4,what's the word for trees in norway,61
...,...,...
5495,find my wallet,42
5496,can you give me the gps location of harvey,42
5497,where's my buddy steve right this second,42
5498,locate jenny at her present position,42


,text,intent
0,"in spanish, meet me tomorrow is said how",61
1,"in french, how do i say, see you later",61
2,how do you say hello in japanese,61
3,how do i ask about the weather in chinese,61
4,"how can i say ""cancel my order"" in french",61
...,...,...
3095,what does it mean to have equality,42
3096,what site publishes the most fake news,42
3097,can you tell me some fake news,42
3098,is cnn really fake news,42


### Data Prepocessing

In [2]:
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC

# nltk.download('punkt') # For word_tokenize
# nltk.download('stopwords') # For the stop_words set
# nltk.download('averaged_perceptron_tagger_eng') # For pos_tag
# nltk.download('wordnet') # For the lemmatizer

def preprocess_sentences(sentences):
    """Preprocess a list of sentences by pos-tagging, tokenizing, removing stop words, and lemmatizing."""
    stop_words = set(stopwords.words('english'))
    lemmatizer = WordNetLemmatizer()
    
    preprocessed_sentences = []
    
    for sentence in sentences:
        tokens = word_tokenize(sentence)
        pos_tags = nltk.pos_tag(tokens)

        cleaned_tokens = [] 

        for token, pos in pos_tags:
            token_lower = token.lower()
            
            # Removes stop words AND drops pure punctuation
            if token_lower not in stop_words and token_lower.isalpha(): 
                if pos.startswith('NN'):
                    lemma = lemmatizer.lemmatize(token_lower, pos='n')
                elif pos.startswith('VB'):
                    lemma = lemmatizer.lemmatize(token_lower, pos='v')
                elif pos.startswith('JJ'):
                    lemma = lemmatizer.lemmatize(token_lower, pos='a')
                else:
                    lemma = lemmatizer.lemmatize(token_lower)
                
                cleaned_tokens.append(lemma)
                
        preprocessed_sentences.append(cleaned_tokens)

    return preprocessed_sentences

#Preprocess the training data
train_text = train_df['text'].tolist()
y_train = train_df['intent'].tolist()
preprocessed_text = preprocess_sentences(train_text)
train_texts = [' '.join(tokens) for tokens in preprocessed_text]

# Preprocess the Validation set
val_texts_raw = validation_df['text'].tolist()
y_val = validation_df['intent'].tolist()
val_preprocessed = preprocess_sentences(val_texts_raw)
val_texts = [" ".join(tokens) for tokens in val_preprocessed]

# Preprocess the Test set
test_texts_raw = test_df['text'].tolist()
y_test = test_df['intent'].tolist()
test_preprocessed = preprocess_sentences(test_texts_raw)
test_texts = [" ".join(tokens) for tokens in test_preprocessed]

### Data Vectorization

In [3]:
# Vectorization (TF-IDF)
vectorizer = TfidfVectorizer()

X_train = vectorizer.fit_transform(train_texts)

# Use transform() on validation and test data. 
# This converts them using the vocabulary learned from the training data.
X_val = vectorizer.transform(val_texts)
X_test = vectorizer.transform(test_texts)

### Model Training (SVM)

In [4]:
print("Training the SVM Model...")
# A 'linear' kernel is almost always the best choice for text classification
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train, y_train)
print("Training Complete!\n")

Training the SVM Model...
Training Complete!



### Model Evaluation

In [5]:
from sklearn.metrics import accuracy_score, classification_report

print("--- Validation Set Results ---")
y_val_pred = svm_model.predict(X_val)
# classification_report shows Precision, Recall, and F1-Score for every single intent!
print(classification_report(y_val, y_val_pred))

print("--- Test Set Results ---")
y_test_pred = svm_model.predict(X_test)
print(f"Final Test Accuracy: {accuracy_score(y_test, y_test_pred) * 100:.2f}%\n")

--- Validation Set Results ---
              precision    recall  f1-score   support

           0       0.85      0.55      0.67        20
           1       0.91      0.50      0.65        20
           2       0.64      0.70      0.67        20
           3       0.74      0.85      0.79        20
           4       0.65      0.85      0.74        20
           5       0.87      0.65      0.74        20
           6       1.00      0.85      0.92        20
           7       0.95      0.95      0.95        20
           8       1.00      0.85      0.92        20
           9       0.64      0.80      0.71        20
          10       0.83      1.00      0.91        20
          11       0.47      0.80      0.59        20
          12       0.95      0.95      0.95        20
          13       0.57      0.65      0.60        20
          14       1.00      1.00      1.00        20
          15       0.94      0.80      0.86        20
          16       0.54      0.75      0.62       

### Model Testing

In [6]:
intent_mapping = {
    0: 'restaurant_reviews', 1: 'nutrition_info', 2: 'account_blocked', 3: 'oil_change_how', 4: 'time', 
    5: 'weather', 6: 'redeem_rewards', 7: 'interest_rate', 8: 'gas_type', 9: 'accept_reservations', 
    10: 'smart_home', 11: 'user_name', 12: 'report_lost_card', 13: 'repeat', 14: 'whisper_mode', 
    15: 'what_are_your_hobbies', 16: 'order', 17: 'jump_start', 18: 'schedule_meeting', 19: 'meeting_schedule', 
    20: 'freeze_account', 21: 'what_song', 22: 'meaning_of_life', 23: 'restaurant_reservation', 24: 'traffic', 
    25: 'make_call', 26: 'text', 27: 'bill_balance', 28: 'improve_credit_score', 29: 'change_language', 
    30: 'no', 31: 'measurement_conversion', 32: 'timer', 33: 'flip_coin', 34: 'do_you_have_pets', 
    35: 'balance', 36: 'tell_joke', 37: 'last_maintenance', 38: 'exchange_rate', 39: 'uber', 
    40: 'car_rental', 41: 'credit_limit', 42: 'oos', 43: 'shopping_list', 44: 'expiration_date', 
    45: 'routing', 46: 'meal_suggestion', 47: 'tire_change', 48: 'todo_list', 49: 'card_declined', 
    50: 'rewards_balance', 51: 'change_accent', 52: 'vaccines', 53: 'reminder_update', 54: 'food_last', 
    55: 'change_ai_name', 56: 'bill_due', 57: 'who_do_you_work_for', 58: 'share_location', 59: 'international_visa', 
    60: 'calendar', 61: 'translate', 62: 'carry_on', 63: 'book_flight', 64: 'insurance_change', 
    65: 'todo_list_update', 66: 'timezone', 67: 'cancel_reservation', 68: 'transactions', 69: 'credit_score', 
    70: 'report_fraud', 71: 'spending_history', 72: 'directions', 73: 'spelling', 74: 'insurance', 
    75: 'what_is_your_name', 76: 'reminder', 77: 'where_are_you_from', 78: 'distance', 79: 'payday', 
    80: 'flight_status', 81: 'find_phone', 82: 'greeting', 83: 'alarm', 84: 'order_status', 
    85: 'confirm_reservation', 86: 'cook_time', 87: 'damaged_card', 88: 'reset_settings', 89: 'pin_change', 
    90: 'replacement_card_duration', 91: 'new_card', 92: 'roll_dice', 93: 'income', 94: 'taxes', 
    95: 'date', 96: 'who_made_you', 97: 'pto_request', 98: 'tire_pressure', 99: 'how_old_are_you', 
    100: 'rollover_401k', 101: 'pto_request_status', 102: 'how_busy', 103: 'application_status', 104: 'recipe', 
    105: 'calendar_update', 106: 'play_music', 107: 'yes', 108: 'direct_deposit', 109: 'credit_limit_change', 
    110: 'gas', 111: 'pay_bill', 112: 'ingredients_list', 113: 'lost_luggage', 114: 'goodbye', 
    115: 'what_can_i_ask_you', 116: 'book_hotel', 117: 'are_you_a_bot', 118: 'next_song', 119: 'change_speed', 
    120: 'plug_type', 121: 'maybe', 122: 'w2', 123: 'oil_change_when', 124: 'thank_you', 125: 'shopping_list_update', 
    126: 'pto_balance', 127: 'order_checks', 128: 'travel_alert', 129: 'fun_fact', 130: 'sync_device', 
    131: 'schedule_maintenance', 132: 'apr', 133: 'transfer', 134: 'ingredient_substitution', 135: 'calories', 
    136: 'current_location', 137: 'international_fees', 138: 'calculator', 139: 'definition', 140: 'next_holiday', 
    141: 'update_playlist', 142: 'mpg', 143: 'min_payment', 144: 'change_user_name', 145: 'restaurant_suggestion', 
    146: 'travel_notification', 147: 'cancel', 148: 'pto_used', 149: 'travel_suggestion', 150: 'change_volume'
}


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

# Create UI Elements
text_input = widgets.Text(
    value='',
    placeholder='Type your sentence here...',
    description='Sentence:',
    layout=widgets.Layout(width='80%')
)
button = widgets.Button(
    description='Predict Intent',
    button_style='success', # Gives it a nice green color
    tooltip='Click to predict the intent'
)
output_area = widgets.Output()

def on_button_clicked(b):
    with output_area:
        clear_output()  # Clear previous output
        
        user_text = text_input.value
        if not user_text:
            print("Please enter a sentence!")
            return
            
        clean_custom = [" ".join(tokens) for tokens in preprocess_sentences([user_text])]
        vectorized_custom = vectorizer.transform(clean_custom)
        prediction_id = svm_model.predict(vectorized_custom)[0]
        
        # Map the ID to the human-readable label
        intent_label = intent_mapping.get(prediction_id, "Unknown Intent")
        
        # Print the beautiful output
        print(f"You typed:      '{user_text}'")
        print(f"Predicted ID:   {prediction_id}")
        print(f"Human Intent:   [{intent_label.upper()}]")

# Link the button to the function
button.on_click(on_button_clicked)

# Display the interface
display(widgets.HBox([text_input, button]))
display(output_area)

Output()